# Faster R-CNN · Cascade R-CNN 결과 비교

두 학습 노트북의 **셀 17-1**에서 `EXPORT_COMPARISON=True`로 저장한 자료를 사용한다.
각 모델에서 선택한 최종 설정의 seed 0·1·2를 같은 함수로 다시 평가한다. 모델 재학습과 Test 추론은 하지 않는다.
별도 프로젝트 Python 파일을 import하지 않는다. 데이터·평가 함수 구현은 이 노트북 상단에 들어 있다.

**순서:** 환경 A → B → 안내 시 커널 재시작 후 A → B → 셀 1 → 내장 함수 → 비교 1~5.
GitHub는 기존 환경 설정과 팀 수정표를 준비하는 데 사용하며, 새 공용 코드의 push가 필요하지 않다.
비교 계산은 기본 커널의 GitHub 고정 버전에서 CPU로 수행하므로 Cascade용 환경 C를 설치하지 않는다.

**비교 대상:** 같은 데이터·분할·학습 예산에서 각 모델이 선택한 최적 설정이다.
공식 프레임워크·COCO 체크포인트·BN eps가 서로 다르므로 이 차이를 설정·환경 표에 함께 기록한다.
추론 시간은 데이터 로딩을 제외한 각 학습 프로세스의 forward 시간이며 GPU·프레임워크 환경과 함께 참고한다.
학습 GPU 메모리는 해당 프로세스의 `max_memory_allocated`다. 다른 프로세스 사용량은 포함하지 않는다.

## 1. 환경 준비·내장 함수

### 환경 A · GitHub 저장소·Python 기준 확인

Colab은 **GPU 런타임**을 먼저 선택한다. 아래 URL·브랜치의 환경 파일을 기준으로 실행한다.
저장소가 없으면 복제하고, 이미 있으면 현재 코드와 커밋을 사용한다.
Python 버전이 다르면 프로젝트 기준 Python 런타임을 선택해야 한다.


In [ ]:
import sys
import os
import json
import hashlib
import importlib
import importlib.metadata as metadata
import shutil
import subprocess
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("현재 Python이 너무 낮습니다. Colab에서 프로젝트 기준 Python 3.12 런타임을 선택하세요.")
import tomllib

REPO_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"
REPO_BRANCH = "exp/23-faster-rcnn-baseline"
# 별도로 업로드한 저장소를 쓰면 해당 루트의 절대 경로를 입력한다.
REPO_ROOT_OVERRIDE = None
IS_COLAB = sys.platform == "linux" and Path("/content").is_dir()

if REPO_ROOT_OVERRIDE:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
elif IS_COLAB:
    REPO_ROOT = Path("/content/pill-object-detection")
else:
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "pyproject.toml").is_file()
                      and (p / "src/pill_detection").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("저장소를 찾지 못했습니다. REPO_ROOT_OVERRIDE에 저장소 루트를 입력하세요.")

# Colab에 저장소가 없을 때만 복제한다. 기존 체크아웃은 자동 갱신하지 않는다.
if IS_COLAB and not REPO_ROOT.exists():
    result = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}, capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError(f"저장소 복제에 실패했습니다. URL·브랜치·접근 권한을 확인하세요.\n{result.stderr[-3000:]}")

reference_names = (".python-version", "pyproject.toml", "uv.lock")
missing = [name for name in reference_names if not (REPO_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"환경 기준 파일이 없습니다: {missing}. 저장소 전체를 준비하고 환경 A부터 다시 실행하세요.")
branch_result = subprocess.run(["git", "branch", "--show-current"], cwd=REPO_ROOT, capture_output=True, text=True)
if branch_result.returncode:
    raise RuntimeError("Git 저장소를 확인할 수 없습니다. 위 URL의 저장소를 복제한 경로를 지정하세요.")
if IS_COLAB and branch_result.stdout.strip() != REPO_BRANCH:
    raise RuntimeError(f"현재 브랜치가 {REPO_BRANCH}와 다릅니다. 작업을 보관한 뒤 지정 브랜치를 준비하세요.")
commit_result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
ENV_REPOSITORY_COMMIT = commit_result.stdout.strip()
ENV_REFERENCE_SHA256 = {name: hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest()
                        for name in reference_names}
python_pin = (REPO_ROOT / ".python-version").read_text().strip()
pin_parts = tuple(int(part) for part in python_pin.split("."))
if tuple(sys.version_info[:len(pin_parts)]) != pin_parts:
    raise RuntimeError(f"프로젝트 기준은 Python {python_pin}, 현재는 {sys.version.split()[0]}입니다. "
                       "Colab에서 해당 Python 런타임을 선택하고 다시 실행하세요. 이 셀은 실행 중인 Python을 교체하지 않습니다.")
if IS_COLAB and Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve():
    raise RuntimeError("Colab 기본 Python 커널을 선택하세요. 학습은 별도 .venv가 아닌 현재 Colab 커널에서 실행합니다.")

print("저장소:", REPO_ROOT)
print("브랜치:", branch_result.stdout.strip(), "| 커밋:", ENV_REPOSITORY_COMMIT[:12])
print("Python:", sys.version.split()[0], "| 현재 커널:", sys.executable)
print("uv.lock SHA256:", ENV_REFERENCE_SHA256["uv.lock"])
print("다음: 환경 B에서 잠금 버전을 적용합니다.")


### 환경 B · uv.lock 설치·커널 재시작 안내

Colab의 **현재 커널**에 `uv sync --locked --no-dev --inexact`를 적용한다.
잠금 버전의 프로젝트 패키지를 설치하며, 잠금 목록 외 Colab 패키지는 유지한다.
설치 변경이 있으면 **현재 VM의 Python 세션/커널만 재시작**하고 환경 A → B부터 다시 실행한다.
재시작 안내가 없어졌을 때 셀 1로 넘어간다. 새 런타임을 삭제·생성하면 설치를 다시 해야 한다.

[uv 잠금·동기화](https://docs.astral.sh/uv/concepts/projects/sync/) ·
[uv 설치 대상 환경](https://docs.astral.sh/uv/concepts/projects/config/#project-environment-path)


In [ ]:
if "ENV_REFERENCE_SHA256" not in globals():
    raise RuntimeError("환경 A를 먼저 실행하세요.")


def installed_versions():
    """현재 커널 환경의 설치 패키지 목록을 반환한다.

    Returns:
        list: 패키지 이름과 버전의 정렬된 목록.
    """
    return sorted((dist.metadata["Name"].lower().replace("_", "-"), dist.version)
                  for dist in metadata.distributions() if dist.metadata.get("Name"))


if not IS_COLAB:
    print("로컬 환경은 자동 변경하지 않습니다. 저장소에서 uv sync --locked 후 해당 .venv 커널을 선택하세요.")
else:
    UV_BIN = shutil.which("uv")
    local_uv = Path.home() / ".local/bin/uv"
    if UV_BIN is None:
        if not local_uv.is_file():
            print("프로젝트 README의 공식 설치 방법으로 uv를 준비합니다.")
            result = subprocess.run(
                ["bash", "-o", "pipefail", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
                capture_output=True, text=True,
            )
            if result.returncode:
                raise RuntimeError(f"uv 설치에 실패했습니다. 네트워크를 확인하고 이 셀을 다시 실행하세요.\n{result.stderr[-3000:]}")
        UV_BIN = str(local_uv)
    packages_before = installed_versions()
    restart_was_required = globals().get("ENV_RESTART_REQUIRED", False)
    # 설치 도중 실패하더라도 바뀐 패키지를 기존 프로세스에서 사용하지 않는다.
    ENV_RESTART_REQUIRED = True
    ENV_SYNC_COMPLETED = False
    print("uv.lock의 버전을 현재 Colab 커널에 설치합니다. 최초 CUDA 패키지 다운로드에는 시간이 걸릴 수 있습니다.")
    result = subprocess.run(
        [UV_BIN, "sync", "--locked", "--no-dev", "--inexact", "--python", sys.executable],
        cwd=REPO_ROOT, env={**os.environ, "UV_PROJECT_ENVIRONMENT": sys.prefix},
        capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError("잠금 환경 설치에 실패했습니다. 아래 원인을 해결한 뒤 커널을 재시작하고 환경 A부터 실행하세요. "
                           f"잠금 파일은 자동 갱신하지 않습니다.\n{result.stderr[-5000:]}")
    if any(hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest() != digest
           for name, digest in ENV_REFERENCE_SHA256.items()):
        raise RuntimeError("설치 중 환경 기준 파일이 바뀌었습니다. 원인을 확인하고 커널 재시작 후 환경 A부터 실행하세요.")
    importlib.invalidate_caches()
    ENV_SYNC_COMPLETED = True
    ENV_RESTART_REQUIRED = restart_was_required or packages_before != installed_versions()
    if ENV_RESTART_REQUIRED:
        print("설치 완료. 다음 셀을 실행하기 전에 Colab 메뉴에서 세션/커널을 재시작하세요.")
        print("런타임 삭제·연결 해제 대신 현재 VM의 Python 세션만 재시작하고, 환경 A → B → 셀 1 순서로 다시 실행하세요.")
    else:
        print("현재 커널의 패키지가 잠금 환경과 같습니다. 셀 1에서 버전·CUDA를 확인하세요.")


### 셀 1 · 비교 환경 버전 검증

커널 재시작 안내가 없어졌다면 실행하고 아래 내장 함수 A~D를 순서대로 실행한다.


In [ ]:
CHECK_GPU_OPS = False  # 저장된 예측 재평가는 CPU로 가능하다.
if "ENV_REFERENCE_SHA256" not in globals():
    raise RuntimeError("환경 A → 환경 B를 먼저 실행하세요.")
if globals().get("ENV_RESTART_REQUIRED", False):
    raise RuntimeError("패키지 설치 후 커널 재시작이 필요합니다. 현재 VM의 Python 세션만 재시작하고 환경 A → B → 셀 1을 실행하세요.")
if IS_COLAB and not globals().get("ENV_SYNC_COMPLETED", False):
    raise RuntimeError("환경 B의 설치가 완료되지 않았습니다. 환경 B를 먼저 실행하세요.")

from packaging.markers import Marker
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name
from packaging.version import Version

project = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text())
locked = tomllib.loads((REPO_ROOT / "uv.lock").read_text())
if Version(sys.version.split()[0]) not in SpecifierSet(project["project"]["requires-python"]):
    raise RuntimeError("현재 Python이 프로젝트 요구 조건과 다릅니다. 환경 A의 Python 기준을 확인하세요.")
if any(hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest() != digest
       for name, digest in ENV_REFERENCE_SHA256.items()):
    raise RuntimeError("환경 기준 파일이 바뀌었습니다. 커널을 재시작하고 환경 A부터 실행하세요.")

# OS별 잠금 항목을 선택한다. +cu130 접미사도 비교하여 CPU 빌드를 걸러낸다.
required_names = {canonicalize_name(Requirement(spec).name)
                  for spec in project["project"]["dependencies"]} | {"pandas"}
version_rows = []
for name in sorted(required_names):
    expected = {p["version"] for p in locked["package"]
                if canonicalize_name(p["name"]) == name and "version" in p
                and (not p.get("resolution-markers")
                     or any(Marker(marker).evaluate() for marker in p["resolution-markers"]))}
    if len(expected) != 1:
        raise RuntimeError(f"현재 OS의 {name} 잠금 버전을 하나로 결정할 수 없습니다. uv.lock과 실행 환경을 확인하세요.")
    expected_version = next(iter(expected))
    try:
        actual_version = metadata.version(name)
    except metadata.PackageNotFoundError:
        actual_version = "미설치"
    version_rows.append({"package": name, "locked": expected_version, "installed": actual_version,
                         "match": actual_version != "미설치" and Version(actual_version) == Version(expected_version)})
mismatches = [row for row in version_rows if not row["match"]]
if mismatches:
    raise RuntimeError(f"잠금 버전과 다른 패키지가 있습니다: {mismatches}. "
                       "Colab은 환경 B, 로컬은 uv sync --locked 실행 후 커널을 재시작하세요.")

# 설치 메타데이터가 맞아도 메모리에는 이전 패키지가 남아 있을 수 있다.
module_names = {"numpy": "numpy", "pandas": "pandas", "pillow": "PIL", "matplotlib": "matplotlib",
                "torch": "torch", "torchvision": "torchvision", "torchmetrics": "torchmetrics", "wandb": "wandb"}
for package, module_name in module_names.items():
    loaded_version = getattr(sys.modules.get(module_name), "__version__", None)
    if loaded_version is not None and Version(loaded_version) != Version(metadata.version(package)):
        raise RuntimeError(f"{package}의 설치 버전과 메모리 버전이 다릅니다. 커널 재시작 후 환경 A부터 실행하세요.")

import torch
import torchvision
from torchvision.ops import nms

if IS_COLAB and CHECK_GPU_OPS:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU를 사용할 수 없습니다. Colab의 하드웨어 가속기를 GPU로 선택하고 다시 연결하세요.")
    try:
        boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], device="cuda")
        scores = torch.tensor([0.9, 0.8], device="cuda")
        keep = nms(boxes, scores, 0.5)
        torch.cuda.synchronize()
        if keep.cpu().tolist() != [0]:
            raise RuntimeError("NMS 결과가 예상과 다릅니다.")
        del boxes, scores, keep
    except Exception as exc:
        raise RuntimeError("CUDA 또는 torchvision GPU 연산 확인에 실패했습니다. GPU/드라이버가 잠금 CUDA 빌드를 지원하는지 확인하세요. "
                           "환경 B 적용 후에도 실패하면 다른 Colab GPU 런타임이 필요합니다.") from exc
    print("CUDA·torchvision NMS GPU 연산: 통과")
else:
    print("비교 노트북: 저장된 예측을 CPU로 재평가합니다. 학습 GPU 검사는 수행하지 않습니다.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ENVIRONMENT = {"python": sys.version.split()[0], "python_executable": sys.executable,
               "packages": {row["package"]: row["installed"] for row in version_rows},
               "cuda": torch.version.cuda,
               "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "uv_lock_sha256": ENV_REFERENCE_SHA256["uv.lock"],
               "environment_repository_commit": ENV_REPOSITORY_COMMIT}
print("Python:", ENVIRONMENT["python"], "| torch CUDA:", ENVIRONMENT["cuda"], "| GPU:", ENVIRONMENT["device"])
display(pd.DataFrame(version_rows))


### 내장 함수 A · bbox·그리기·코드 식별자

In [ ]:
# 이 셀부터 함수 구현을 노트북 안에 직접 정의한다. 별도 프로젝트 모듈을 읽지 않는다.
import ast
import hashlib
import inspect
import json
import linecache
import textwrap
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_CODE_BLOCKS = {}


def _register_notebook_block(name):
    """실행한 셀의 소스를 기록하여 재개 검증과 Cascade 작업 전달에 사용한다.

    Args:
        name (str): 이 노트북 안의 코드 묶음 이름.
    """
    frame = inspect.currentframe().f_back
    try:
        source = "".join(linecache.getlines(frame.f_code.co_filename))
        if not source:
            shell = get_ipython()
            source = shell.history_manager.input_hist_raw[-1]
        if not source.strip():
            raise RuntimeError("실행 셀의 소스를 찾을 수 없습니다. Jupyter/Colab에서 내장 함수 셀부터 다시 실행하세요.")
        NOTEBOOK_CODE_BLOCKS[name] = source
    finally:
        del frame


def _notebook_source_hash(names):
    """셀 번호·주석을 제외한 실제 내장 코드의 식별자를 계산한다.

    Args:
        names (list): 해시할 코드 묶음 이름.

    Returns:
        str: Python 3.11과 3.12에서 공통으로 비교할 수 있는 소스 해시.
    """
    entries = {}
    for name in names:
        if name not in NOTEBOOK_CODE_BLOCKS:
            raise RuntimeError(f"내장 함수 셀이 빠졌습니다: {name}. 위에서부터 함수 셀을 실행하세요.")
        tree = ast.parse(NOTEBOOK_CODE_BLOCKS[name])
        for node in ast.walk(tree):
            # 이 코드에서는 제네릭 타입 문법을 쓰지 않으므로 3.12의 빈 필드만 제외한다.
            if hasattr(node, "type_params") and not node.type_params:
                del node.type_params
        entries[name] = ast.dump(tree, include_attributes=False)
    return hashlib.sha256(json.dumps(entries, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def common_code_identity():
    """세 노트북에 같은 내용으로 들어간 데이터·평가 함수의 해시를 반환한다.

    Returns:
        str: 비교에 사용할 공통 소스 식별자.
    """
    return _notebook_source_hash(["helpers", "data", "analysis"])


def xyxy_to_xywh(bbox):
    """Ultralytics 추론 결과의 xyxy bbox를 xywh로 바꾼다.

    Args:
        bbox (list[float]): [좌상단 x, 좌상단 y, 우하단 x, 우하단 y]. 원본 픽셀.

    Returns:
        list[float]: [좌상단 x, 좌상단 y, 너비, 높이]. 원본 픽셀.
    """
    x1, y1, x2, y2 = bbox
    return [x1, y1, x2 - x1, y2 - y1]

def draw_boxes(image, boxes, labels=None, color="red", width=4):
    """이미지에 xywh bbox를 그려 새 PIL 이미지로 돌려준다.

    기본 글꼴은 한글을 그리지 못하므로 labels에는 category_id 같은 숫자·영문을 넣는다.

    Args:
        image (str | Path | PIL.Image.Image): 이미지 경로나 PIL 이미지.
        boxes (list[list[float]]): 그릴 bbox 목록. 각각 [좌상단 x, 좌상단 y, 너비, 높이], 원본 픽셀.
        labels (list | None): bbox마다 위에 적을 글자. boxes와 순서가 같다. None이면 적지 않는다.
        color (str): 선과 글자의 색.
        width (int): 선 두께(픽셀).

    Returns:
        PIL.Image.Image: bbox를 그린 RGB 이미지.
    """
    image = Image.open(image) if not isinstance(image, Image.Image) else image
    image = image.convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=28)
    for i, (x, y, w, h) in enumerate(boxes):
        draw.rectangle([x, y, x + w, y + h], outline=color, width=width)
        if labels:
            draw.text((x, max(0, y - 32)), str(labels[i]), fill=color, font=font)
    return image

_register_notebook_block("helpers")

### 내장 함수 B · 데이터·고정 분할·수정 라벨

In [ ]:
"""알약 검출의 고정 분할·수정 라벨·원본 좌표를 관리한다."""

import ast
import hashlib
import json
import os
import shutil
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor


def file_sha256(path):
    """파일 내용을 해시한다.

    Args:
        path (str | Path): 검사할 파일.

    Returns:
        str: SHA-256.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """JSON으로 표현 가능한 설정의 식별자를 만든다.

    Args:
        value (object): 해시할 값.

    Returns:
        str: SHA-256.
    """
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """임시 파일을 완성한 뒤 원자적으로 JSON을 저장한다.

    Args:
        path (str | Path): 저장 위치.
        value (object): JSON 값.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(text)
    os.replace(temporary, path)


def _bbox(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    array = np.asarray(value, dtype=float)
    if array.shape != (4,) or not np.isfinite(array).all() or (array[2:] <= 0).any():
        raise ValueError("bbox는 유한한 xywh 숫자 4개와 양수 너비·높이여야 합니다. 라벨표를 수정하세요.")
    return array.tolist()


def prepare_data(image_dir, annotation_dir, split_path, fixes_path):
    """원본은 보존하고 고정 분할과 add/modify 수정표를 적용한다.

    Args:
        image_dir (str | Path): 학습 이미지 폴더.
        annotation_dir (str | Path): COCO JSON 폴더.
        split_path (str | Path): file_name, combo, split 열의 고정 분할.
        fixes_path (str | Path): file_name, category_id, action, bbox 열의 수정표.

    Returns:
        dict: 이미지 목록·클래스 매핑·객체 라벨·데이터 식별자.
    """
    image_dir, annotation_dir = Path(image_dir), Path(annotation_dir)
    split_path, fixes_path = Path(split_path), Path(fixes_path)
    for path in (image_dir, annotation_dir, split_path, fixes_path):
        if not path.exists():
            raise FileNotFoundError(f"데이터 경로가 없습니다: {path}. 노트북 1번의 경로를 지정하세요.")
    json_paths = sorted(annotation_dir.rglob("*.json"))
    if not json_paths:
        raise ValueError("annotation JSON이 없습니다. train_annotations 경로를 확인하세요.")
    images, annotations, names, source_hashes = {}, [], {}, []
    for path in json_paths:
        doc = json.loads(path.read_text(encoding="utf-8-sig"))
        local_images = {int(item["id"]): item for item in doc["images"]}
        for item in local_images.values():
            name = item["file_name"]
            if Path(name).name != name:
                raise ValueError("이미지 파일명에 경로가 포함됐습니다. 이미지 manifest를 확인하세요.")
            record = {"file_name": name, "image_id": int(item["id"]),
                      "width": int(item["width"]), "height": int(item["height"])}
            if name in images and images[name] != record:
                raise ValueError(f"같은 이미지의 ID·크기가 다릅니다: {name}. 원본 연결을 확인하세요.")
            images[name] = record
        for category in doc.get("categories", []):
            names[int(category["id"])] = str(category.get("name") or category["id"])
        for ann in doc.get("annotations", []):
            if int(ann["image_id"]) not in local_images:
                raise ValueError(f"JSON 내부 image_id가 연결되지 않습니다: {path}. 라벨을 확인하세요.")
            annotations.append({"file_name": local_images[int(ann["image_id"])]["file_name"],
                                "category_id": int(ann["category_id"]), "bbox": _bbox(ann["bbox"]),
                                "iscrowd": int(ann.get("iscrowd", 0))})
        source_hashes.append((str(path.relative_to(annotation_dir)), file_sha256(path)))
    if not annotations:
        raise ValueError("객체 라벨이 없습니다. annotation 내용을 확인하세요.")
    raw_count = len(annotations)
    fixes = pd.read_csv(fixes_path, keep_default_na=False)
    if "bbox" not in fixes.columns and "bbox_after" in fixes.columns:
        fixes = fixes.rename(columns={"bbox_after": "bbox"})
    required = {"file_name", "category_id", "action", "bbox"}
    if not required.issubset(fixes.columns):
        raise ValueError(f"수정표에 {sorted(required)} 열이 필요합니다. label_fixes.csv를 확인하세요.")
    changed = Counter()
    for row in fixes.to_dict("records"):
        name, category = str(row["file_name"]).strip(), int(row["category_id"])
        if name not in images:
            raise ValueError(f"수정할 이미지가 원본에 없습니다: {name}. 수정표를 확인하세요.")
        bbox, action = _bbox(row["bbox"]), str(row["action"]).strip().lower()
        # 팀 공유표의 bbox·area도 기존 객체 수정으로 정규화한다.
        action = {"bbox": "modify", "area": "modify"}.get(action, action)
        matches = [a for a in annotations if a["file_name"] == name and a["category_id"] == category]
        if action == "add":
            if any(np.allclose(a["bbox"], bbox, atol=1e-6, rtol=0) for a in matches):
                raise ValueError(f"추가 라벨이 중복됩니다: {name}, {category}. 수정표를 확인하세요.")
            annotations.append({"file_name": name, "category_id": category, "bbox": bbox, "iscrowd": 0})
        elif action == "modify":
            if len(matches) != 1:
                raise ValueError(f"수정 대상이 {len(matches)}개입니다: {name}, {category}. 라벨을 구분하세요.")
            matches[0]["bbox"] = bbox
        else:
            raise ValueError(f"지원하지 않는 수정 action: {action}. add·modify·bbox·area로 지정하세요.")
        changed[action] += 1
    frame = pd.DataFrame(sorted(images.values(), key=lambda r: r["file_name"]))
    if frame["image_id"].duplicated().any():
        raise ValueError("서로 다른 이미지에 같은 image_id가 있습니다. 원본 연결을 확인하세요.")
    split = pd.read_csv(split_path, dtype=str, keep_default_na=False)
    if not {"file_name", "combo", "split"}.issubset(split.columns):
        raise ValueError("분할표에 file_name·combo·split 열이 필요합니다. 고정 분할표를 확인하세요.")
    if split["file_name"].duplicated().any() or set(split["file_name"]) != set(frame["file_name"]):
        raise ValueError("분할표의 이미지 목록이 원본과 다릅니다. 누락·중복을 확인하세요.")
    if set(split["split"]) != {"train", "val"}:
        raise ValueError("분할에는 train·val이 모두 있어야 합니다. 고정 분할표를 확인하세요.")
    if not split["combo"].eq(split["file_name"].str.split("_").str[0]).all():
        raise ValueError("분할표의 combo와 파일명 조합이 다릅니다. 고정 분할표를 확인하세요.")
    if split.groupby("combo")["split"].nunique().max() > 1:
        raise ValueError("같은 조합이 train·val에 섞였습니다. 팀의 고정 분할표를 확인하세요.")
    frame = frame.merge(split, on="file_name", validate="one_to_one")
    image_hashes = []
    for row in frame.to_dict("records"):
        path = image_dir / row["file_name"]
        with Image.open(path) as image:
            if image.size != (row["width"], row["height"]):
                raise ValueError(f"이미지와 라벨의 크기가 다릅니다: {path}. 좌표 기준을 확인하세요.")
        image_hashes.append((row["file_name"], file_sha256(path)))
    hash_splits = defaultdict(set)
    for name, digest in image_hashes:
        hash_splits[digest].add(split.set_index("file_name").loc[name, "split"])
    if any(len(parts) > 1 for parts in hash_splits.values()):
        raise ValueError("동일한 이미지 내용이 train·val에 중복됩니다. 고정 분할을 검토하세요.")
    categories = sorted({a["category_id"] for a in annotations})
    mapping = {category: index + 1 for index, category in enumerate(categories)}
    objects = defaultdict(list)
    for ann in annotations:
        info = images[ann["file_name"]]
        x, y, w, h = ann["bbox"]
        if x < 0 or y < 0 or x + w > info["width"] or y + h > info["height"]:
            raise ValueError(f"경계 밖 bbox: {ann['file_name']}, {ann['category_id']}. 수정표를 확인하세요.")
        if ann["iscrowd"] != 0:
            raise ValueError("이 실험은 개별 알약 iscrowd=0을 전제로 합니다. 평가 정책을 확인하세요.")
        xyxy = box_convert(torch.tensor([ann["bbox"]], dtype=torch.float64), "xywh", "xyxy")[0].tolist()
        objects[ann["file_name"]].append({"boxes": xyxy, "category_id": ann["category_id"],
                                          "label": mapping[ann["category_id"]]})
    class_map = pd.DataFrame({"category_id": categories,
                              "model_label": [mapping[c] for c in categories],
                              "name": [names.get(c, str(c)) for c in categories]})
    identity = stable_hash({"images": image_hashes, "annotations": source_hashes,
                            "split": file_sha256(split_path), "fixes": file_sha256(fixes_path),
                            "mapping": mapping})
    return {"frame": frame, "objects": dict(objects), "class_map": class_map,
            "category_to_label": mapping, "label_to_category": {v: k for k, v in mapping.items()},
            "image_dir": image_dir, "identity": identity, "image_hashes": dict(image_hashes),
            "raw_boxes": raw_count, "fixed_boxes": len(annotations), "fixes": dict(changed)}


def data_summary(bundle):
    """분할별 이미지 수와 클래스별 정답 객체 수를 집계한다.

    Args:
        bundle (dict): prepare_data 결과.

    Returns:
        tuple: 분할 요약표, 클래스별 객체 수.
    """
    rows = []
    for row in bundle["frame"].to_dict("records"):
        for obj in bundle["objects"].get(row["file_name"], []):
            rows.append({"split": row["split"], "category_id": obj["category_id"]})
    counts = pd.crosstab(pd.Series([r["category_id"] for r in rows], name="category_id"),
                         pd.Series([r["split"] for r in rows], name="split"))
    counts = counts.reindex(index=bundle["class_map"]["category_id"], columns=["train", "val"], fill_value=0)
    summary = pd.DataFrame([{"split": part, "images": int(bundle["frame"]["split"].eq(part).sum()),
                             "boxes": int(counts[part].sum()), "classes": int((counts[part] > 0).sum())}
                            for part in ("train", "val")])
    return summary, counts.reset_index()


def cache_images(bundle, cache_root):
    """내용 해시가 일치하는 로컬 이미지 캐시를 준비한다.

    Args:
        bundle (dict): 데이터 묶음.
        cache_root (str | Path): 복사본 저장 루트.

    Returns:
        dict: image_dir만 캐시 경로로 바꾼 묶음.
    """
    destination = Path(cache_root) / bundle["identity"][:12]
    destination.mkdir(parents=True, exist_ok=True)
    for name, digest in bundle["image_hashes"].items():
        target = destination / name
        if not target.exists() or file_sha256(target) != digest:
            with tempfile.NamedTemporaryFile(dir=destination, delete=False) as stream:
                temporary = Path(stream.name)
            shutil.copyfile(bundle["image_dir"] / name, temporary)
            if file_sha256(temporary) != digest:
                raise IOError("이미지 복사본이 원본과 다릅니다. 캐시를 다시 준비하세요.")
            os.replace(temporary, target)
    return {**bundle, "image_dir": destination}


class PillDataset(Dataset):
    """원본 픽셀 이미지와 torchvision용 정답을 반환한다."""

    def __init__(self, frame, image_dir, objects=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.objects = objects

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row["file_name"]) as image:
            tensor = to_tensor(image.convert("RGB"))
        objects = [] if self.objects is None else self.objects.get(row["file_name"], [])
        boxes = torch.tensor([o["boxes"] for o in objects], dtype=torch.float32).reshape(-1, 4)
        return tensor, {"boxes": boxes, "labels": torch.tensor([o["label"] for o in objects], dtype=torch.int64),
                        "image_id": torch.tensor(int(row["image_id"])),
                        "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
                        "iscrowd": torch.zeros(len(objects), dtype=torch.int64)}


def collate_detection(batch):
    """객체 수가 다른 이미지와 정답을 목록으로 묶는다.

    Args:
        batch (list): 이미지·정답 쌍.

    Returns:
        tuple: 이미지 목록, 정답 목록.
    """
    return tuple(zip(*batch))


def make_dataset(bundle, split):
    """지정된 고정 분할의 Dataset을 만든다.

    Args:
        bundle (dict): 데이터 묶음.
        split (str): train 또는 val.

    Returns:
        PillDataset: 선택 분할.
    """
    return PillDataset(bundle["frame"].loc[bundle["frame"]["split"].eq(split)],
                       bundle["image_dir"], bundle["objects"])


def cache_dataset(dataset, cache_root):
    """Test를 포함한 Dataset의 이미지를 검증된 로컬 캐시로 연결한다.

    Args:
        dataset (PillDataset): 복사할 이미지 목록.
        cache_root (str | Path): 캐시 루트.

    Returns:
        PillDataset: 같은 순서·정답을 유지하는 로컬 Dataset.
    """
    hashes = {name: file_sha256(dataset.image_dir / name) for name in dataset.frame["file_name"]}
    source = {"image_dir": dataset.image_dir, "image_hashes": hashes,
              "identity": stable_hash({"files": hashes, "frame": dataset.frame.to_dict("records")})}
    cached = cache_images(source, cache_root)
    return PillDataset(dataset.frame, cached["image_dir"], dataset.objects)


def test_dataset(image_dir):
    """숫자 파일명을 대회 image_id로 보존한 Test Dataset을 만든다.

    Args:
        image_dir (str | Path): Test 이미지 폴더.

    Returns:
        PillDataset: 정답이 없는 이미지 목록.
    """
    image_dir = Path(image_dir)
    rows = []
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        if not path.stem.isdigit():
            raise ValueError(f"Test 파일명이 숫자가 아닙니다: {path.name}. 대회 파일을 확인하세요.")
        with Image.open(path) as image:
            width, height = image.size
        rows.append({"file_name": path.name, "image_id": int(path.stem), "width": width, "height": height})
    frame = pd.DataFrame(rows)
    if frame.empty or frame["image_id"].duplicated().any():
        raise ValueError("Test 이미지가 없거나 image_id가 중복됩니다. 경로·파일명을 확인하세요.")
    return PillDataset(frame.sort_values("image_id"), image_dir)


_register_notebook_block("data")

### 내장 함수 C · 공통 mAP·시각화·CSV

In [ ]:
"""모델에 독립적인 원본 좌표 평가·오류 분석·제출 변환."""

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou, box_convert




IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
SUBMISSION_COLUMNS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y",
                      "bbox_w", "bbox_h", "score"]


def evaluate_records(records, class_map=None, detailed=True):
    """공통 예측의 bbox mAP와 선택적 클래스·IoU별 AP를 계산한다.

    Args:
        records (list): prediction과 target을 포함한 이미지별 결과. labels는 원래 category ID.
        class_map (DataFrame | None): 전체 클래스 목록.
        detailed (bool): 상세 결과 계산 여부.

    Returns:
        dict: 요약, 클래스별 AP, IoU별 AP, PR 곡선 원자료.
    """
    if not records or any("target" not in row for row in records):
        raise ValueError("정답이 있는 validation 결과가 필요합니다. 4번 검증 추론을 실행하세요.")
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=IOU_THRESHOLDS,
                                  max_detection_thresholds=[1, 10, 100], class_metrics=detailed,
                                  extended_summary=detailed)
    metric.update([row["prediction"] for row in records], [row["target"] for row in records])
    result = metric.compute()
    summary = {"map_75_95": float(result["map"]), "map_75": float(result["map_75"]),
               "mar_100": float(result["mar_100"]), "images": len(records),
               "ground_truths": sum(len(r["target"]["boxes"]) for r in records)}
    if not detailed:
        return {"summary": summary}
    counts = Counter(int(c) for row in records for c in row["target"]["labels"])
    precision = result["precision"][:, :, :, 0, -1].numpy()
    classes = result["classes"].reshape(-1).tolist()

    def valid_mean(values):
        valid = values[values >= 0]
        return float(valid.mean()) if valid.size else None

    ap = pd.DataFrame([{"category_id": int(category), "ap_75_95": valid_mean(precision[:, :, index]),
                        "val_objects": counts[int(category)]} for index, category in enumerate(classes)])
    if class_map is not None:
        ap = class_map[["category_id", "name"]].merge(ap, how="left", on="category_id")
        ap["val_objects"] = ap["val_objects"].fillna(0).astype(int)
    ap.loc[ap["val_objects"].eq(0), "ap_75_95"] = np.nan
    iou_ap = pd.DataFrame({"iou": IOU_THRESHOLDS,
                           "ap": [valid_mean(precision[index]) for index in range(len(IOU_THRESHOLDS))]})
    return {"summary": summary, "classes": ap, "ious": iou_ap,
            "precision": precision, "category_ids": classes}


def detection_errors(records, categories, iou_threshold=0.75, score_threshold=0.30):
    """클래스와 독립적인 IoU 일대일 대응으로 혼동과 미검출을 집계한다.

    Args:
        records (list): validation 공통 결과.
        categories (list): 원래 category ID 목록.
        iou_threshold (float): 대응 최소 IoU.
        score_threshold (float): 진단용 최소 score.

    Returns:
        tuple: 행=정답·열=예측 혼동행렬, 오류 상세 표.
    """
    if not 0 < iou_threshold <= 1 or not 0 <= score_threshold <= 1:
        raise ValueError("진단 IoU·score 범위가 잘못됐습니다. 4번 설정을 확인하세요.")
    categories = [int(c) for c in categories]
    positions = {category: index for index, category in enumerate(categories)}
    background = len(categories)
    matrix = np.zeros((background + 1, background + 1), dtype=int)
    events = []
    for record in records:
        target, pred = record["target"], record["prediction"]
        kept = torch.where(pred["scores"] >= score_threshold)[0]
        boxes, labels, scores = pred["boxes"][kept], pred["labels"][kept], pred["scores"][kept]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        candidates = np.argwhere(overlaps >= iou_threshold)
        candidates = sorted(candidates.tolist(), key=lambda pair: (-overlaps[tuple(pair)], pair))
        used_gt, used_pred = set(), set()

        def append(kind, gt=None, prediction=None, overlap=None):
            events.append({"image_id": int(record["image_id"]), "file_name": record["file_name"],
                           "kind": kind, "gt_index": gt,
                           "pred_index": None if prediction is None else int(kept[prediction]),
                           "true_category": None if gt is None else int(target["labels"][gt]),
                           "pred_category": None if prediction is None else int(labels[prediction]),
                           "score": None if prediction is None else float(scores[prediction]),
                           "iou": overlap})

        for gt, prediction in candidates:
            if gt in used_gt or prediction in used_pred:
                continue
            used_gt.add(gt)
            used_pred.add(prediction)
            truth, estimate = int(target["labels"][gt]), int(labels[prediction])
            matrix[positions[truth], positions[estimate]] += 1
            append("correct" if truth == estimate else "class_confusion", gt, prediction,
                   float(overlaps[gt, prediction]))
        for gt in range(len(target["boxes"])):
            if gt not in used_gt:
                matrix[positions[int(target["labels"][gt])], background] += 1
                possible = [p for p in range(len(boxes)) if int(labels[p]) == int(target["labels"][gt])
                            and 0 < overlaps[gt, p] < iou_threshold]
                best = max(possible, key=lambda p: overlaps[gt, p]) if possible else None
                append("localization_or_missed" if best is not None else "missed", gt, best,
                       None if best is None else float(overlaps[gt, best]))
        for prediction in range(len(boxes)):
            if prediction not in used_pred:
                matrix[background, positions[int(labels[prediction])]] += 1
                duplicate = any(int(target["labels"][gt]) == int(labels[prediction])
                                and overlaps[gt, prediction] >= iou_threshold for gt in used_gt)
                append("duplicate" if duplicate else "extra", prediction=prediction)
    labels = [str(c) for c in categories] + ["unmatched"]
    columns = ["image_id", "file_name", "kind", "gt_index", "pred_index", "true_category",
               "pred_category", "score", "iou"]
    return pd.DataFrame(matrix, index=labels, columns=labels), pd.DataFrame(events, columns=columns)


def plot_ground_truth(dataset, reverse, count=3):
    """입력 데이터의 정답 bbox를 표시한다.

    Args:
        dataset (PillDataset): 정답이 있는 이미지.
        reverse (dict): 모델 번호에서 원래 클래스 ID로 가는 매핑.
        count (int): 표시할 이미지 수.

    Returns:
        Figure: 정답 박스 이미지.
    """
    fig, axes = plt.subplots(1, min(count, len(dataset)), figsize=(6 * min(count, len(dataset)), 6), squeeze=False)
    for index, axis in enumerate(axes[0]):
        _, target = dataset[index]
        image = draw_boxes(dataset.image_dir / dataset.frame.iloc[index]["file_name"],
                           [xyxy_to_xywh(box.tolist()) for box in target["boxes"]],
                           [str(reverse[int(c)]) for c in target["labels"]], color="lime")
        axis.imshow(image)
        axis.set(title=dataset.frame.iloc[index]["file_name"][:30])
        axis.axis("off")
    fig.tight_layout()
    return fig


def plot_learning_curves(run_dirs):
    """완료 여부와 관계없이 저장된 학습 곡선을 표시한다.

    Args:
        run_dirs (list): metrics.csv를 포함한 실행 폴더.

    Returns:
        Figure: loss·mAP 곡선.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for directory in run_dirs:
        path = Path(directory) / "metrics.csv"
        if not path.exists():
            continue
        history = pd.read_csv(path)
        if history.empty:
            continue
        name = Path(directory).name
        axes[0].plot(history["epoch"], history["train_loss"], label=name)
        axes[1].plot(history["epoch"], history["val_map_75_95"], label=name)
    for axis, title in zip(axes, ["Train loss", "Validation mAP@[.75:.95]"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend(fontsize=7)
    fig.tight_layout()
    return fig


def plot_candidate_scores(results):
    """설정별 seed 점수와 평균·표본 표준편차를 함께 표시한다.

    Args:
        results (DataFrame): 완료된 실행 결과.

    Returns:
        Figure: seed별 점수와 설정별 평균. 단일 seed의 표준편차는 추정하지 않음.
    """
    summary = results.groupby("candidate_id")["best_val_map_75_95"].agg(["mean", "std", "count"])
    summary = summary.sort_values("mean", ascending=False)
    positions = {key: index for index, key in enumerate(summary.index)}
    x = np.arange(len(summary))
    fig, axis = plt.subplots(figsize=(max(8, len(summary)*1.2), 5))
    axis.bar(x, summary["mean"], color="lightsteelblue", label="Mean")
    estimated = summary["count"].ge(2).to_numpy()
    axis.errorbar(x[estimated], summary.loc[estimated, "mean"], yerr=summary.loc[estimated, "std"],
                  fmt="none", ecolor="black", capsize=4, label="Sample SD (n>=2)")
    for seed, group in results.groupby("seed"):
        axis.scatter([positions[key] for key in group["candidate_id"]], group["best_val_map_75_95"],
                     label=f"seed {int(seed)}", zorder=3)
    axis.set(xticks=x, xticklabels=[f"{key}\nn={int(summary.loc[key, 'count'])}" for key in summary.index],
             ylabel="Validation mAP@[.75:.95]", ylim=(0, 1.05),
             title="Candidate and seed comparison (n=1: SD unavailable)")
    axis.tick_params(axis="x", rotation=30, labelsize=8)
    axis.legend(fontsize=8)
    axis.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    return fig


def plot_ap(report):
    """낮은 클래스 AP·정답 수와 IoU별 AP를 함께 표시한다.

    Args:
        report (dict): 상세 evaluate_records 결과.

    Returns:
        Figure: 클래스·IoU별 그래프.
    """
    low = report["classes"].dropna(subset=["ap_75_95"]).nsmallest(15, "ap_75_95")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh([f"{int(row.category_id)} (n={int(row.val_objects)})" for row in low.itertuples()], low["ap_75_95"])
    axes[0].invert_yaxis()
    axes[0].set(title="Lowest class AP (validation support)", xlabel="AP@[.75:.95]", xlim=(0, 1))
    axes[1].plot(report["ious"]["iou"], report["ious"]["ap"], marker="o")
    axes[1].set(title="AP by IoU", xlabel="IoU", ylabel="AP", ylim=(0, 1))
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    return fig


def plot_confusion(matrix, normalize=True):
    """정답 행 기준 정규화 또는 원시 건수 혼동행렬을 표시한다.

    Args:
        matrix (DataFrame): 행=정답·열=예측. unmatched 행은 추가 예측, 열은 미검출.
        normalize (bool): 정답 행별 비율 표시 여부.

    Returns:
        Figure: 진단 혼동행렬.
    """
    values = matrix.to_numpy(dtype=float)
    if normalize:
        values = np.divide(values, values.sum(axis=1, keepdims=True),
                           out=np.zeros_like(values), where=values.sum(axis=1, keepdims=True) > 0)
    size = max(9, min(20, len(matrix) * 0.28))
    fig, axis = plt.subplots(figsize=(size, size))
    heatmap = axis.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    axis.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
             xticklabels=matrix.columns, yticklabels=matrix.index,
             xlabel="Predicted (unmatched = missed)", ylabel="True (unmatched = extra)",
             title="Detection confusion: row-normalized" if normalize else "Detection confusion: counts")
    axis.tick_params(axis="x", rotation=90, labelsize=7)
    axis.tick_params(axis="y", labelsize=7)
    fig.colorbar(heatmap, ax=axis, fraction=0.04)
    fig.tight_layout()
    return fig


def preview_records(records, events=None, count=6, score_threshold=0.30):
    """정답·예측과 오류 종류를 표시한다. Test에는 정답을 그리지 않는다.

    Args:
        records (list): 공통 예측 결과.
        events (DataFrame | None): validation 오류 표.
        count (int): 최대 이미지 수.
        score_threshold (float): 표시용 점수 하한.

    Returns:
        Figure: 초록 정답·빨강 예측 이미지.
    """
    if events is not None:
        errors = events.loc[events["kind"].ne("correct")].copy()
        errors = errors.sort_values("score", ascending=False, na_position="last")
        ids = list(dict.fromkeys(errors["image_id"].tolist()))[:count]
        chosen = [r for identifier in ids for r in records if r["image_id"] == identifier]
    else:
        chosen = records[:count]
    if not chosen:
        fig, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, "No error images at the chosen diagnostic thresholds", ha="center")
        axis.axis("off")
        return fig
    fig, axes = plt.subplots(len(chosen), 1, figsize=(9, 6 * len(chosen)), squeeze=False)
    for record, axis in zip(chosen, axes[:, 0]):
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
        if "target" in record:
            target = record["target"]
            image = draw_boxes(image, [xyxy_to_xywh(b.tolist()) for b in target["boxes"]],
                               [f"GT:{int(c)}" for c in target["labels"]], color="lime")
        pred = record["prediction"]
        indices = torch.where(pred["scores"] >= score_threshold)[0]
        indices = indices[torch.argsort(pred["scores"][indices], descending=True)][:20]
        captions = []
        for index in indices:
            caption = f"P:{int(pred['labels'][index])} {float(pred['scores'][index]):.2f}"
            if "target" in record and len(record["target"]["boxes"]):
                overlap = float(box_iou(pred["boxes"][index].reshape(1, 4), record["target"]["boxes"]).max())
                caption += f" I:{overlap:.2f}"
            captions.append(caption)
        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in indices],
                           captions, color="red")
        image.thumbnail((1280, 1280))
        kinds = "prediction only; no ground truth"
        if events is not None:
            kinds = ", ".join(sorted(set(events.loc[events["image_id"].eq(record["image_id"]), "kind"])))
        axis.imshow(image)
        axis.set_title(f"ID {record['image_id']} | {kinds}\ngreen=GT, red=prediction; score>={score_threshold}", fontsize=9)
        axis.axis("off")
    fig.tight_layout()
    return fig


def submission_frame(records, allowed_categories):
    """공통 원본 좌표 예측을 대회 CSV 표로 변환한다.

    Args:
        records (list): labels가 원래 category ID인 공통 예측.
        allowed_categories (list): 허용 클래스 목록.

    Returns:
        DataFrame: 대회 컬럼 순서의 표. 빈 예측도 빈 표로 표현한다.
    """
    allowed_categories = set(int(c) for c in allowed_categories)
    if len({int(r["image_id"]) for r in records}) != len(records):
        raise ValueError("예측 이미지 ID가 중복됩니다. 전체 추론 목록을 확인하세요.")
    rows = []
    for record in records:
        pred = record["prediction"]
        if not (len(pred["boxes"]) == len(pred["labels"]) == len(pred["scores"])):
            raise ValueError("예측 boxes·labels·scores 수가 다릅니다. 공통 변환을 확인하세요.")
        for box, category, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
            x, y, w, h = xyxy_to_xywh(box.tolist())
            if int(category) not in allowed_categories:
                raise ValueError("알 수 없는 category ID입니다. 모델의 클래스 매핑을 확인하세요.")
            if not np.isfinite([x, y, w, h, float(score)]).all() or not 0 <= float(score) <= 1:
                raise ValueError("유한하지 않은 좌표·score가 있습니다. 모델 출력을 확인하세요.")
            if x < -1e-4 or y < -1e-4 or w <= 0 or h <= 0 or x+w > record["width"]+1e-4 or y+h > record["height"]+1e-4:
                raise ValueError("원본 경계 밖 또는 면적 0 예측입니다. 좌표 복원을 확인하세요.")
            rows.append({"image_id": int(record["image_id"]), "category_id": int(category),
                         "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h, "score": float(score)})
    frame = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])
    frame = frame.sort_values(["image_id", "score"], ascending=[True, False], kind="stable").reset_index(drop=True)
    frame.insert(0, "annotation_id", np.arange(1, len(frame)+1, dtype=np.int64))
    for column in ("annotation_id", "image_id", "category_id"):
        frame[column] = frame[column].astype("int64")
    return frame[SUBMISSION_COLUMNS]


def restore_submission(frame, records):
    """CSV를 다시 읽은 뒤 이미지별 공통 예측으로 복원한다.

    Args:
        frame (DataFrame): 제출 컬럼의 표.
        records (list): 전체 이미지·정답 목록.

    Returns:
        list: 빈 이미지도 보존한 복원 결과.
    """
    if list(frame.columns) != SUBMISSION_COLUMNS or frame.isna().any().any():
        raise ValueError("제출 컬럼·결측값이 잘못됐습니다. CSV 생성 코드를 확인하세요.")
    try:
        frame = frame.apply(pd.to_numeric, errors="raise")
    except (TypeError, ValueError) as error:
        raise ValueError("CSV에 숫자가 아닌 값이 있습니다. 제출 파일을 다시 생성하세요.") from error
    if not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV에 유한하지 않은 값이 있습니다. 제출 파일을 다시 생성하세요.")
    for column in ("annotation_id", "image_id", "category_id"):
        if not np.equal(frame[column], np.floor(frame[column])).all():
            raise ValueError("CSV ID는 정수여야 합니다. 클래스·이미지 매핑을 확인하세요.")
    if frame["annotation_id"].duplicated().any() or not set(frame["image_id"]).issubset({r["image_id"] for r in records}):
        raise ValueError("제출 ID가 중복되거나 이미지 목록과 다릅니다. CSV를 확인하세요.")
    restored = []
    for record in records:
        rows = frame.loc[frame["image_id"].eq(record["image_id"])]
        boxes = torch.tensor(rows[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy(dtype=np.float32), dtype=torch.float32).reshape(-1, 4)
        restored.append({**record, "prediction": {
            "boxes": box_convert(boxes, "xywh", "xyxy"),
            "labels": torch.tensor(rows["category_id"].to_numpy(dtype=np.int64), dtype=torch.int64),
            "scores": torch.tensor(rows["score"].to_numpy(dtype=np.float32), dtype=torch.float32)}})
    return restored


def verify_roundtrip(records, categories, csv_path):
    """실제 CSV 저장·읽기와 전후 mAP·좌표·ID 일치를 검사한다.

    Args:
        records (list): 정답을 포함한 validation 예측.
        categories (list): 원래 클래스 ID 목록.
        csv_path (str | Path): 검증용 CSV 저장 위치.

    Returns:
        dict: 변환 전후 mAP와 통과 결과.
    """
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = submission_frame(records, categories)
    frame.to_csv(path, index=False)
    loaded = pd.read_csv(path)
    restored = restore_submission(loaded, records)
    for before, after in zip(records, restored):
        left, right = before["prediction"], after["prediction"]
        order = torch.argsort(left["scores"], descending=True, stable=True)
        if not torch.equal(left["labels"][order], right["labels"]) or not torch.allclose(left["boxes"][order], right["boxes"], atol=1e-3, rtol=0) or not torch.allclose(left["scores"][order], right["scores"], atol=1e-7, rtol=0):
            raise ValueError("CSV 복원 전후 예측이 바뀌었습니다. ID·좌표 변환을 확인하세요.")
    before = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
    after = evaluate_records(restored, detailed=False)["summary"]["map_75_95"]
    if abs(before-after) > 1e-6:
        raise ValueError("CSV 변환 전후 mAP가 다릅니다. 평가 입력과 좌표 변환을 확인하세요.")
    report = {"before": before, "after": after, "passed": True, "images": len(records), "rows": len(frame)}
    write_json(path.with_suffix(".json"), report)
    return report


_register_notebook_block("analysis")

### 내장 함수 D · 결과 검증·두 모델 비교

In [ ]:
MODEL_KEYS = {"faster": "faster_rcnn_r50_fpn", "cascade": "cascade_rcnn_r50_fpn"}


def discover_final_groups(root):
    """최종 seed 0·1·2 설정이 선택된 결과 폴더를 찾는다.

    Args:
        root (str | Path): 모델별 runs 루트 또는 결과 그룹 폴더.

    Returns:
        list: final_selection.json을 포함한 폴더 목록.
    """
    root = Path(root)
    paths = ([root / "final_selection.json"] if (root / "final_selection.json").is_file()
             else sorted(root.glob("*/final_selection.json")))
    return [path.parent for path in paths]


def load_comparison_group(model_key, directory, bundle):
    """같은 데이터와 공통 코드로 생성한 세 seed의 validation 예측만 읽는다.

    Args:
        model_key (str): faster 또는 cascade.
        directory (str | Path): 해당 모델의 최종 설정 그룹 폴더.
        bundle (dict): 현재 원본·고정 분할·수정 라벨.

    Returns:
        list: 검사한 실행별 manifest·예측·실제 폴더.
    """
    directory = Path(directory)
    selected = json.loads((directory / "final_selection.json").read_text())
    runs = []
    validation = bundle["frame"].loc[bundle["frame"]["split"].eq("val")]
    expected_images = {int(row.image_id): row for row in validation.itertuples()}
    for saved_path in selected["run_dirs"]:
        saved = Path(saved_path)
        # Drive의 마운트 위치가 바뀌어도 선택한 그룹 안의 같은 실행 폴더를 우선 사용한다.
        run_dir = directory / saved.name
        if not run_dir.is_dir():
            raise FileNotFoundError(f"선택한 그룹에 실행 폴더가 없습니다: {run_dir}. 모델별 저장 경로를 확인하세요.")
        manifest_path = run_dir / "comparison_manifest.json"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"비교 자료가 없습니다: {manifest_path}. 해당 학습 노트북 셀 17-1에서 EXPORT_COMPARISON=True로 실행하세요.")
        manifest = json.loads(manifest_path.read_text())
        original = json.loads((run_dir / "config.json").read_text())
        if manifest.get("format") != "pill_detection_notebook_comparison_v1":
            raise ValueError("비교 자료 형식이 다릅니다. 새 학습 노트북의 셀 17-1에서 다시 저장하세요.")
        if manifest["model"] != MODEL_KEYS[model_key] or manifest["spec"]["candidate_id"] != selected["candidate_id"]:
            raise ValueError("선택한 모델·최종 후보와 실행 내용이 다릅니다. 결과 그룹을 확인하세요.")
        if manifest["data_identity"] != bundle["identity"] or manifest["common_code"] != common_code_identity():
            raise ValueError("학습 때의 분할·수정 라벨·이미지 또는 공통 함수가 현재와 다릅니다. 같은 데이터·함수 버전의 결과를 준비하세요.")
        if manifest["class_map"] != bundle["class_map"].to_dict("records"):
            raise ValueError("클래스 매핑이 다릅니다. 두 학습 노트북의 클래스 목록을 확인하세요.")
        signature = stable_hash({"recipe": original["recipe"], "spec": original["spec"]})
        if original["signature"] != signature or manifest["run_signature"] != signature:
            raise ValueError("실행 설정과 비교 자료의 식별자가 다릅니다. 같은 실행에서 자료를 다시 저장하세요.")
        if manifest["spec"] != original["spec"] or manifest["config"] != original["recipe"]["config"]:
            raise ValueError("실험 조건과 비교 자료가 다릅니다. 셀 17-1의 내보내기를 다시 실행하세요.")
        if manifest["iou_thresholds"] != IOU_THRESHOLDS or manifest["max_detections"] != 100:
            raise ValueError("평가 IoU·최대 검출 수가 다릅니다. 동일 평가 정책의 결과를 사용하세요.")
        if manifest["prediction_file"] != "validation_predictions.pt":
            raise ValueError("validation 예측 파일명이 다릅니다. 정상적인 비교 자료를 다시 저장하세요.")
        prediction_path = run_dir / manifest["prediction_file"]
        if file_sha256(prediction_path) != manifest["prediction_sha256"]:
            raise ValueError("비교 자료 저장 뒤 예측 파일이 바뀌었습니다. 셀 17-1에서 다시 내보내세요.")
        payload = torch.load(prediction_path, map_location="cpu", weights_only=False)
        records = payload["records"]
        if len(records) != len(expected_images) or {r["image_id"] for r in records} != set(expected_images):
            raise ValueError("validation 이미지가 누락·중복됐습니다. 전체 validation 예측을 다시 생성하세요.")
        for record in records:
            info = expected_images[record["image_id"]]
            if (record["file_name"], record["width"], record["height"]) != (info.file_name, info.width, info.height):
                raise ValueError("validation 이미지의 파일명·원본 크기가 다릅니다. 예측 연결을 확인하세요.")
            objects = bundle["objects"].get(info.file_name, [])
            truth_boxes = torch.tensor([o["boxes"] for o in objects], dtype=torch.float32).reshape(-1, 4)
            truth_labels = torch.tensor([o["category_id"] for o in objects], dtype=torch.int64)
            truth = record.get("target", {})
            if ("boxes" not in truth or "labels" not in truth or
                    not torch.equal(truth["boxes"], truth_boxes) or not torch.equal(truth["labels"], truth_labels)):
                raise ValueError("저장된 validation 정답이 현재 라벨과 다릅니다. 같은 수정 라벨로 다시 추론하세요.")
            record["image_path"] = str(bundle["image_dir"] / info.file_name)
        submission_frame(records, bundle["class_map"]["category_id"])
        runs.append({"model": model_key, "run_dir": run_dir, "manifest": manifest, "records": records})
    if sorted(r["manifest"]["spec"]["seed"] for r in runs) != [0, 1, 2]:
        raise ValueError("각 모델에 seed 0·1·2가 정확히 하나씩 필요합니다. 최종 설정의 반복 학습을 완료하세요.")
    return sorted(runs, key=lambda run: run["manifest"]["spec"]["seed"])


def validate_comparison_policies(groups):
    """모델별 최적 설정을 비교할 때 유지해야 하는 공통 학습 예산을 확인한다.

    Args:
        groups (dict): faster·cascade의 검사된 실행 목록.
    """
    if set(groups) != set(MODEL_KEYS):
        raise ValueError("Faster와 Cascade 결과가 모두 필요합니다. 두 결과 그룹을 선택하세요.")
    keys = ("max_epochs", "patience", "image_size", "batch_size", "optimizer", "precision", "augmentation",
            "weight_decay", "betas", "warmup_epochs", "final_lr_ratio", "learning_rates", "max_detections",
            "score_threshold", "nms_iou")
    configurations = [{key: run["manifest"]["config"][key] for key in keys}
                      for runs in groups.values() for run in runs]
    if any(config != configurations[0] for config in configurations[1:]):
        raise ValueError("두 모델의 고정 조건·탐색 범위가 다릅니다. 같은 학습 예산의 결과를 선택하세요.")


def reevaluate_comparison(groups, class_map):
    """두 모델의 모든 seed를 현재 커널의 동일 mAP 함수로 다시 평가한다.

    Args:
        groups (dict): 검사된 두 모델의 실행 목록.
        class_map (DataFrame): 약 클래스 목록.

    Returns:
        tuple: seed별 점수표, (모델, seed)별 상세 평가.
    """
    validate_comparison_policies(groups)
    rows, reports = [], {}
    for model_key, runs in groups.items():
        for run in runs:
            doc = run["manifest"]
            report = evaluate_records(run["records"], class_map, detailed=True)
            seed = int(doc["spec"]["seed"])
            reports[(model_key, seed)] = report
            rows.append({"model": model_key, "seed": seed, "candidate_id": doc["spec"]["candidate_id"],
                         "mode": doc["spec"]["mode"], "lr": doc["spec"]["lr"],
                         **report["summary"], "stored_best_map": doc["result"]["best_val_map_75_95"],
                         "training_seconds": doc["result"]["training_seconds"],
                         "inference_ms": doc["speed"]["model_ms_per_image"],
                         "peak_gpu_gb": doc["result"]["peak_gpu_gb"], "gpu": doc["runtime"]["device"],
                         "framework": doc["runtime"]["framework"], "python": doc["runtime"]["python"],
                         "torch": doc["runtime"]["packages"]["torch"],
                         "normalization": doc["config"]["normalization"], "run_dir": str(run["run_dir"])})
    scores = pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)
    return scores, reports


def plot_two_model_scores(scores):
    """각 seed의 mAP와 세 seed의 평균·표본 표준편차를 그린다.

    Args:
        scores (DataFrame): 재평가한 seed별 점수.

    Returns:
        Figure: 두 모델의 점수 비교.
    """
    fig, axis = plt.subplots(figsize=(8, 5))
    for position, model in enumerate(MODEL_KEYS):
        rows = scores.loc[scores["model"].eq(model)]
        mean, deviation = rows["map_75_95"].mean(), rows["map_75_95"].std(ddof=1)
        axis.bar(position, mean, yerr=deviation, capsize=5, alpha=0.5)
        axis.scatter([position - 0.1, position, position + 0.1], rows.sort_values("seed")["map_75_95"],
                     color="black", zorder=3)
    axis.set(xticks=[0, 1], xticklabels=["Faster R-CNN", "Cascade R-CNN"], ylim=(0, 1.05),
             ylabel="Validation mAP@[.75:.95]", title="Seeds 0, 1, 2; mean and sample SD")
    axis.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    return fig


def plot_two_model_images(groups, image_ids, seed=0, score_threshold=0.30):
    """같은 이미지·같은 seed의 정답과 두 모델 예측을 나란히 표시한다.

    Args:
        groups (dict): 검사된 두 모델의 실행 목록.
        image_ids (list): 같이 볼 validation 이미지 ID.
        seed (int): 시각화에 사용할 seed.
        score_threshold (float): 표시용 최소 점수. mAP 계산에는 적용하지 않음.

    Returns:
        Figure: 초록 정답·빨강 예측 비교.
    """
    if not image_ids:
        raise ValueError("비교할 이미지 ID가 없습니다. validation 이미지에서 ID를 선택하세요.")
    if len(set(image_ids)) != len(image_ids):
        raise ValueError("이미지 ID가 중복됐습니다. 서로 다른 이미지를 선택하세요.")
    fig, axes = plt.subplots(len(image_ids), 2, figsize=(13, 5 * len(image_ids)), squeeze=False)
    for column, model in enumerate(MODEL_KEYS):
        run = next(r for r in groups[model] if r["manifest"]["spec"]["seed"] == seed)
        lookup = {r["image_id"]: r for r in run["records"]}
        for row, image_id in enumerate(image_ids):
            if image_id not in lookup:
                plt.close(fig)
                raise ValueError(f"validation에 없는 이미지 ID입니다: {image_id}. 이미지 목록을 확인하세요.")
            record = lookup[image_id]
            truth, prediction = record["target"], record["prediction"]
            image = draw_boxes(record["image_path"], [xyxy_to_xywh(b.tolist()) for b in truth["boxes"]],
                               [f"GT:{int(c)}" for c in truth["labels"]], color="lime")
            kept = torch.where(prediction["scores"] >= score_threshold)[0]
            kept = kept[torch.argsort(prediction["scores"][kept], descending=True)][:20]
            image = draw_boxes(image, [xyxy_to_xywh(prediction["boxes"][i].tolist()) for i in kept],
                               [f"P:{int(prediction['labels'][i])} {float(prediction['scores'][i]):.2f}" for i in kept], color="red")
            image.thumbnail((1000, 1000))
            axes[row, column].imshow(image)
            axes[row, column].set_title(f"{model} | seed {seed} | ID {image_id} | score >= {score_threshold}")
            axes[row, column].axis("off")
    fig.tight_layout()
    return fig

## 2. 데이터와 결과 선택

### 비교 1 · 데이터·학습 결과·비교 결과 경로

In [ ]:
IS_COLAB = Path("/content").is_dir() and "google.colab" in sys.modules
if Path("/content").is_dir():
    try:
        from google.colab import drive
        IS_COLAB = True
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except ImportError:
        pass
DATA_ROOT = (Path("/content/drive/MyDrive/코드잇 ;)/Project1_pill-object-detection/data")
             if IS_COLAB else REPO_ROOT / "data")
IMAGE_DIR = DATA_ROOT / "raw/train_images"
ANNOTATION_DIR = DATA_ROOT / "raw/train_annotations"
SPLIT_PATH = DATA_ROOT / "processed/split.csv"
FIXES_PATH = DATA_ROOT / "processed/label_fixes.csv"
if not FIXES_PATH.is_file():
    FIXES_PATH = REPO_ROOT / "notebooks/eda/label_fix/jaeone_label_fixes.csv"
RESULT_BASE = DATA_ROOT.parent if IS_COLAB else REPO_ROOT
MODEL_RESULT_ROOTS = {"faster": RESULT_BASE / "runs/cj_faster_experiments",
                      "cascade": RESULT_BASE / "runs/cj_cascade_experiments"}
COMPARISON_ROOT = RESULT_BASE / "runs/cj_two_stage_comparison"
bundle = prepare_data(IMAGE_DIR, ANNOTATION_DIR, SPLIT_PATH, FIXES_PATH)
print("데이터 식별자:", bundle["identity"])
display(data_summary(bundle)[0])
display(pd.DataFrame([{"model": name, "result_root": str(path)} for name, path in MODEL_RESULT_ROOTS.items()]))

### 비교 2 · 각 모델의 최종 설정 그룹 선택

각 모델에서 최종 설정 그룹이 하나면 자동 선택한다. 여러 개라면 아래 `SELECTED_GROUP_DIRS`에 경로를 넣는다.
각 학습 노트북에서 최종 설정의 seed 0·1·2를 완료하고 셀 17-1의 비교 자료 저장도 실행해야 한다.

In [ ]:
SELECTED_GROUP_DIRS = {"faster": None, "cascade": None}
available = {model: discover_final_groups(root) for model, root in MODEL_RESULT_ROOTS.items()}
display(pd.DataFrame([{"model": model, "group_dir": str(path)}
                      for model, paths in available.items() for path in paths], columns=["model", "group_dir"]))
groups = {}
for model in MODEL_KEYS:
    choice = SELECTED_GROUP_DIRS[model]
    if choice is None and len(available[model]) == 1:
        choice = available[model][0]
    if choice is None:
        print(model, "최종 설정이 없거나 여러 개입니다. 학습을 완료하거나 SELECTED_GROUP_DIRS에 결과 그룹을 지정하세요.")
        continue
    groups[model] = load_comparison_group(model, choice, bundle)
    print(model, "선택:", choice)

## 3. 동일 지표로 재평가

### 비교 3 · seed별 mAP·평균·표준편차·환경 비교

In [ ]:
scores, reports = pd.DataFrame(), {}
if set(groups) == set(MODEL_KEYS):
    scores, reports = reevaluate_comparison(groups, bundle["class_map"])
    COMPARISON_DIR = COMPARISON_ROOT / stable_hash({
        "data": bundle["identity"], "common_code": common_code_identity(),
        "runs": {model: [r["manifest"]["run_signature"] for r in runs] for model, runs in groups.items()},
    })[:12]
    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    summary = scores.groupby("model").agg(
        seed_count=("seed", "nunique"), map_mean=("map_75_95", "mean"), map_std=("map_75_95", "std"),
        training_seconds_mean=("training_seconds", "mean"), inference_ms_mean=("inference_ms", "mean"),
        process_peak_gpu_gb=("peak_gpu_gb", "max"),
    )
    paired = scores.pivot(index="seed", columns="model", values="map_75_95")
    paired["cascade_minus_faster"] = paired["cascade"] - paired["faster"]
    display(scores[["model", "seed", "candidate_id", "map_75_95", "map_75", "mar_100", "stored_best_map"]])
    display(summary)
    display(paired)
    display(scores[["model", "seed", "gpu", "framework", "python", "torch", "normalization"]])
    if scores["gpu"].nunique() != 1:
        print("학습 GPU가 서로 다릅니다. 시간·메모리 수치는 GPU 정보와 함께 확인하세요.")
    scores.to_csv(COMPARISON_DIR / "seed_scores.csv", index=False)
    summary.to_csv(COMPARISON_DIR / "model_summary.csv")
    paired.to_csv(COMPARISON_DIR / "paired_seed_scores.csv")
    write_json(COMPARISON_DIR / "comparison_settings.json", {
        "data_identity": bundle["identity"], "common_code": common_code_identity(),
        "iou_thresholds": IOU_THRESHOLDS, "evaluation_environment": ENVIRONMENT,
        "run_dirs": {model: [str(r["run_dir"]) for r in runs] for model, runs in groups.items()},
        "comparison": "independently_selected_configurations_under_equal_search_budget",
    })
    figure = plot_two_model_scores(scores)
    figure.savefig(COMPARISON_DIR / "model_seed_scores.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("비교 결과:", COMPARISON_DIR)
else:
    print("두 모델의 최종 설정과 비교 자료가 준비되면 비교 2부터 다시 실행하세요.")

### 비교 4 · 클래스별·IoU별 AP 차이

In [ ]:
if reports:
    class_rows, iou_rows = [], []
    for (model, seed), report in reports.items():
        class_rows.append(report["classes"].assign(model=model, seed=seed))
        iou_rows.append(report["ious"].assign(model=model, seed=seed))
    class_scores = pd.concat(class_rows, ignore_index=True)
    iou_scores = pd.concat(iou_rows, ignore_index=True)
    class_mean = class_scores.groupby(["category_id", "name", "model"], dropna=False)["ap_75_95"].mean().unstack("model")
    class_mean["cascade_minus_faster"] = class_mean["cascade"] - class_mean["faster"]
    iou_mean = iou_scores.groupby(["iou", "model"])["ap"].mean().unstack("model")
    display(class_mean.sort_values("cascade_minus_faster"))
    display(iou_mean)
    class_scores.to_csv(COMPARISON_DIR / "class_seed_ap.csv", index=False)
    class_mean.to_csv(COMPARISON_DIR / "class_mean_ap_comparison.csv")
    iou_scores.to_csv(COMPARISON_DIR / "iou_seed_ap.csv", index=False)
    figure, axis = plt.subplots(figsize=(7, 4))
    iou_mean.plot(ax=axis, marker="o")
    axis.set(xlabel="IoU", ylabel="AP", ylim=(0, 1), title="Mean AP across seeds by IoU")
    axis.grid(alpha=0.2)
    figure.tight_layout()
    figure.savefig(COMPARISON_DIR / "iou_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)
else:
    print("비교 3의 재평가 결과가 필요합니다.")

## 4. 같은 이미지에서 오류 확인

### 비교 5 · 혼동행렬·오류 유형·동일 이미지 예측

시각화는 기본 seed 0으로 고정한다. 점수가 가장 높은 seed를 골라 대표 이미지로 삼지 않는다.
표시용 score 0.30·IoU 0.75는 진단에만 사용하며 mAP 재평가에는 별도 score 필터를 적용하지 않는다.

In [ ]:
VISUAL_SEED = 0
DIAGNOSTIC_SCORE = 0.30
DIAGNOSTIC_IOU = 0.75
IMAGE_IDS = []  # 빈 목록이면 두 모델에서 오류가 있는 공통 validation 이미지의 ID 순으로 최대 3개.
if reports:
    error_tables = []
    for model, runs in groups.items():
        chosen_run = next(r for r in runs if r["manifest"]["spec"]["seed"] == VISUAL_SEED)
        matrix, errors = detection_errors(chosen_run["records"], bundle["class_map"]["category_id"],
                                           iou_threshold=DIAGNOSTIC_IOU, score_threshold=DIAGNOSTIC_SCORE)
        error_tables.append(errors.assign(model=model))
        matrix.to_csv(COMPARISON_DIR / f"{model}_seed{VISUAL_SEED}_confusion.csv")
        figure = plot_confusion(matrix, normalize=True)
        figure.savefig(COMPARISON_DIR / f"{model}_seed{VISUAL_SEED}_confusion.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)
    all_errors = pd.concat(error_tables, ignore_index=True)
    display(pd.crosstab(all_errors["model"], all_errors["kind"]))
    all_errors.to_csv(COMPARISON_DIR / f"seed{VISUAL_SEED}_error_events.csv", index=False)
    if not IMAGE_IDS:
        IMAGE_IDS = sorted(set(all_errors.loc[all_errors["kind"].ne("correct"), "image_id"]))[:3]
        if not IMAGE_IDS:
            IMAGE_IDS = sorted(r["image_id"] for r in groups["faster"][0]["records"])[:3]
    figure = plot_two_model_images(groups, IMAGE_IDS, seed=VISUAL_SEED, score_threshold=DIAGNOSTIC_SCORE)
    figure.savefig(COMPARISON_DIR / f"seed{VISUAL_SEED}_same_images.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    write_json(COMPARISON_DIR / "visual_settings.json", {
        "seed": VISUAL_SEED, "image_ids": [int(i) for i in IMAGE_IDS],
        "score": DIAGNOSTIC_SCORE, "iou": DIAGNOSTIC_IOU,
    })
else:
    print("비교 3의 재평가를 먼저 실행하세요.")